# B 分片多步载荷 + 原生终态仅同步状态：固定 clean 用户运行包

本轮已采纳固定单一缓存宿主与35槽方案，并授权发布本包，真实结果待运行。
由人类在GPU上直接Run all执行一次固定clean实验；agent不执行真实35槽。没有执行模式开关。
B方法：多步分片payload；原生终态只有22pilot，绝不再写用户消息或parity。
首包复用既有源01的4×8分片terminal和uint8 native，该缓存不是独立新来源，生成/DiT/梯度=0；
代码另有可调用native50步分片适配路径，但当前clean runner/notebook不调用它，仅有stub分派验证，不能宣称新B真实轨迹或真实梯度验证。
VAEload=1，D=1（只S），E=2，state读取8、segment FFT/read8，共35槽；失败耗槽，后续Run all只读。
P为旧缓存RGB8、S为当前新D，质量是pipeline增量，可能混入VAE环境差异，非纯state效应。
仅clean公开网格，无估计路径，不构成盲同步闭环；不运行phase2编辑同步，不引入A残差clock/CENTERED/U，不宣称同步增益或科学PASS。
旧state-only状态1/22负结果保留；旧22/22来自state+payload，不是此组合成功。

配套源码ZIP与notebook一同发布到现有dev分支，Run all默认通过普通URL自动下载源码ZIP，无需手传。
SOURCE_LOCATION也可直接填写可用源码目录、ZIP或普通URL。
源码与输入均按实际可读性处理，不要求hash/manifest/Git状态/版本相等。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
OUTPUT=Path('/content/drive/MyDrive/Video-WM/B-Segmented-Native-Terminal-Sync-V1/fixed-clean-pair-v1')
def execution_artifacts(output):
    output=Path(output)
    names=("attempt.json","result.json","checkpoint.json","ledger.json","protocol.json","construction.json",
           "state_completion.json","blind_completion.json","evaluation.json")
    found=[n+s for n in names for s in ("",".tmp") if (output/(n+s)).exists() or (output/(n+s)).is_symlink()]
    for folder in ("blind","features","media","quality","latent_difference"):
        root=output/folder
        if root.is_dir():found.extend(str(p.relative_to(output)) for p in root.rglob("*") if p.is_file() or p.is_symlink())
    return sorted(found)
EXISTING_ARTIFACTS=execution_artifacts(OUTPUT)
if EXISTING_ARTIFACTS:
    print('Existing attempt: saved records only',EXISTING_ARTIFACTS)
else:
    import stat,urllib.request,zipfile
    SOURCE_LOCATION='https://raw.githubusercontent.com/RICHAAARC/SC-SSTW/dev/local-joint-state-payload-v1/notebooks/local_segmented_terminal_sync_v1_portable_source.zip'
    source=Path(SOURCE_LOCATION)
    if source.is_dir():SOURCE_ROOT=source
    else:
        archive=source
        if SOURCE_LOCATION.startswith(('https://','http://')):
            archive=Path('/content/b_segmented_state_source.zip');urllib.request.urlretrieve(SOURCE_LOCATION,archive)
        SOURCE_ROOT=Path('/content/b_segmented_state_source');SOURCE_ROOT.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(archive) as bundle:
            for info in bundle.infolist():
                target=(SOURCE_ROOT/info.filename).resolve()
                if not target.is_relative_to(SOURCE_ROOT.resolve()) or stat.S_ISLNK(info.external_attr>>16):
                    raise ValueError('unsafe source archive path')
            bundle.extractall(SOURCE_ROOT)
    if not (SOURCE_ROOT/'experiments/wan_state_clock/local_segmented_terminal_sync_v1_run.py').is_file():raise FileNotFoundError('actual B runner missing')
    OUTPUT.mkdir(parents=True,exist_ok=True)
    print('Using available source',SOURCE_ROOT)


依赖与固定一次运行：使用已有Wan VAE缓存；只修复实际缺包/已知导入冲突。固定输出已存在attempt时跳过安装和模型调用。

In [ ]:

import subprocess, sys

EXISTING_ARTIFACTS = execution_artifacts(OUTPUT)
if not EXISTING_ARTIFACTS:
    LOG = OUTPUT/"setup.log"
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode

    # Copied from the already-used A/P2 setup; no A import/build dependency.
    def ensure_imports(probe, packages, label):
        import re
        module_packages = {package.replace('-', '_'): package for package in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], ('Recheck ' if repaired else 'Probe ')+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named ['\"]([^'\"]+)['\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1) in module_packages:
                package = module_packages[missing.group(1)]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' failed without a known targeted repair; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, 'Repair '+label+': '+action)

    ensure_imports('import numpy,torch', ['numpy', 'torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub; from diffusers import AutoencoderKLWan',
                   ['diffusers', 'accelerate', 'safetensors', 'huggingface-hub'], 'Wan VAE imports')
    def ensure_media_tools():
        import shutil
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            command(['apt-get', '-qq', 'update'], 'Prepare media dependency')
            command(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], 'Install FFmpeg')
        if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
            raise RuntimeError('ffmpeg/ffprobe still unavailable; see setup.log')
    ensure_media_tools()

    with (OUTPUT/"runner.log").open("w") as log:
        process = subprocess.Popen([sys.executable, "-u", "-m",
            "experiments.wan_state_clock.local_segmented_terminal_sync_v1_run"],
            cwd=SOURCE_ROOT, stdout=log, stderr=subprocess.STDOUT)
    import time
    try:
        while process.poll() is None:
            time.sleep(30)
            ledger_path = OUTPUT/"ledger.json"
            if ledger_path.exists():
                ledger = json.loads(ledger_path.read_text())
                returned = sum(row["status"] == "RETURNED" for row in ledger.values())
                active = [name for name, row in ledger.items() if row["status"] == "STARTED"]
                print(f"Completed core units: {returned}/{len(ledger)}; active: {active}")
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        print("Stopped; saved STARTED calls remain unresolved and will not be replayed.")
        raise
    print("Runner return code:", process.returncode)
else:
    print("Existing attempt: displaying saved records; no physical calls repeated.")
print("Output:", OUTPUT)


In [ ]:

saved={}
for name in ('result.json','checkpoint.json','ledger.json','evaluation.json','attempt.json'):
    path=OUTPUT/name
    if path.is_file():
        try:saved[name]=json.loads(path.read_text())
        except (OSError,ValueError) as exc:print('Unreadable saved record',name,type(exc).__name__,str(exc))
r=saved.get('result.json',saved.get('checkpoint.json',{}))
print('Execution',r.get('status'),'Fixed denominator',r.get('fixed_denominator'),'Calls',r.get('counts'))
for slot,v in r.get('evaluation',{}).items():
    print(slot,{k:{n:v[k][n] for n in ('denominator','correct','errors','erasures','unavailable')} for k in ('state','payload')})
    print('Local segments',v.get('segments'))
print('Paired common bits, coverage and regressions',r.get('paired_readout'))
print('Actual terminal delta',r.get('latent_difference'))
for layer,q in r.get('quality',{}).items():
    print('Pipeline quality',layer,{k:v for k,v in q.items() if k not in ('frames','adjacent','peaks')})
    print('All maximum ties',q.get('peaks'))
for name,v in saved.get('ledger.json',{}).items():
    if v.get('status')!='RETURNED':print('Retained failed/unentered slot',name,v)
print('Clean coexistence only; no edit synchronization or scientific PASS. Output:',OUTPUT)
